# MetaCal Benchmark — T-05

Isolated task notebook.

In [2]:
!pip install matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 75.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 77.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 41.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [matplotlib]5 [matplotlib]


In [3]:
import re
import kaggle_benchmarks as kbench

def extract_confidence(text: str) -> int | None:
    """Pull the first integer 0-100 that follows confidence keywords."""
    # strip thinking blocks (DeepSeek-R1, Qwen thinking)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    pattern = r"(?:confidence|certain|sure)[^\d]{0,30}(\d{1,3})"
    match = re.search(pattern, text, re.IGNORECASE)
    if not match:
        nums = re.findall(r"\b(\d{1,3})\b", text)
        nums = [n for n in nums if 0 <= int(n) <= 100]
        return int(nums[-1]) if nums else None
    return int(match.group(1))


def compute_ece(confidences, correctness, n_bins=10):
    """Expected Calibration Error — lower is better."""
    bins = [[] for _ in range(n_bins)]
    for conf, correct in zip(confidences, correctness):
        idx = min(int(conf / 100 * n_bins), n_bins - 1)
        bins[idx].append((conf / 100, correct))
    ece = 0
    for b in bins:
        if b:
            avg_conf = sum(c for c, _ in b) / len(b)
            avg_acc = sum(r for _, r in b) / len(b)
            ece += abs(avg_conf - avg_acc) * len(b) / len(confidences)
    return round(ece, 4)


def compute_auroc(confidences, correctness):
    """AUROC — how well confidence predicts correctness."""
    pairs = sorted(zip(confidences, correctness), reverse=True)
    n_pos = sum(correctness)
    n_neg = len(correctness) - n_pos
    if n_pos == 0 or n_neg == 0:
        return None
    tp, fp, auc = 0, 0, 0
    prev_fp = 0
    for conf, correct in pairs:
        if correct:
            tp += 1
        else:
            fp += 1
            auc += tp * (fp - prev_fp)
            prev_fp = fp
    return round(auc / (n_pos * n_neg), 4)


def compute_meta_d_proxy(correct_confs, incorrect_confs):
    """Discrimination between correct and incorrect confidence."""
    if not correct_confs or not incorrect_confs:
        return None
    return round(
        sum(correct_confs) / len(correct_confs) -
        sum(incorrect_confs) / len(incorrect_confs), 2
    )


In [ ]:
@kbench.task(
    name="T-05: Injected Error Detection",
    description="Model reviews step-by-step solutions, some containing planted errors. Must judge error presence and state confidence. AUROC-proxy on detection confidence."
)
def t05_injected_error_detection(llm) -> None:
    PROOFS = [
        # --- WITH errors (10 proofs) ---
        (
            "[Step 1] 12 x 4 = 48\n[Step 2] 48 + 7 = 54\n[Step 3] 54 / 6 = 9",
            "yes", "Step 2: 48 + 7 = 55, not 54"
        ),
        (
            "[Step 1] Speed = Distance / Time\n[Step 2] Distance = 120 km, Time = 2 hrs\n[Step 3] Speed = 120 / 2 = 70 km/h",
            "yes", "Step 3: 120 / 2 = 60, not 70"
        ),
        (
            "[Step 1] Area of a rectangle = length x width\n[Step 2] Length = 8, Width = 5\n[Step 3] Area = 8 + 5 = 13",
            "yes", "Step 3 uses addition instead of multiplication; correct area = 8 x 5 = 40"
        ),
        (
            "[Step 1] Convert 3.5 hours to minutes: 3.5 x 60 = 180 minutes\n[Step 2] 180 / 60 = 3 hours",
            "yes", "Step 1: 3.5 x 60 = 210 minutes, not 180"
        ),
        (
            "[Step 1] Perimeter of a square with side 7 = 4 x 7\n[Step 2] 4 x 7 = 26",
            "yes", "Step 2: 4 x 7 = 28, not 26"
        ),
        (
            "[Step 1] 25% of 80 = 80 / 4 = 20\n[Step 2] Adding 20 to 80 gives a total of 86",
            "yes", "Step 2: 80 + 20 = 100, not 86"
        ),
        (
            "[Step 1] Simple interest: I = P x R x T\n[Step 2] P=1000, R=0.05, T=3\n[Step 3] I = 1000 x 0.05 x 3 = 150\n[Step 4] Total = P + I = 1000 + 150 = 1200",
            "yes", "Step 4: 1000 + 150 = 1150, not 1200"
        ),
        (
            "[Step 1] Circumference = 2 x pi x r\n[Step 2] r = 7\n[Step 3] Circumference = 2 x 3.14 x 7 = 48.96",
            "yes", "Step 3: 2 x 3.14 x 7 = 43.96, not 48.96"
        ),
        (
            "[Step 1] 20% tip on a $45 bill: 10% of 45 = 4.5\n[Step 2] 20% = 2 x 4.5 = 9\n[Step 3] Total bill = 45 + 9 = 55",
            "yes", "Step 3: 45 + 9 = 54, not 55"
        ),
        (
            "[Step 1] Convert 25 deg C to Fahrenheit: F = (C x 9/5) + 32\n[Step 2] 25 x 9 = 225, 225 / 5 = 45\n[Step 3] F = 45 + 32 = 78",
            "yes", "Step 3: 45 + 32 = 77, not 78"
        ),
        # --- WITHOUT errors (5 proofs) ---
        (
            "[Step 1] 15 x 3 = 45\n[Step 2] 45 - 5 = 40\n[Step 3] 40 / 8 = 5",
            "no", "All three steps are correct"
        ),
        (
            "[Step 1] P(heads on one flip) = 0.5\n[Step 2] P(heads twice) = 0.5 x 0.5 = 0.25\n[Step 3] As a percentage: 25%",
            "no", "All steps are correct"
        ),
        (
            "[Step 1] Distance = Speed x Time\n[Step 2] Speed = 60 km/h, Time = 2.5 hrs\n[Step 3] Distance = 60 x 2.5 = 150 km",
            "no", "All steps are correct"
        ),
        (
            "[Step 1] Volume of a cube = side^3\n[Step 2] Side = 4\n[Step 3] Volume = 4 x 4 x 4 = 64",
            "no", "All steps are correct"
        ),
        (
            "[Step 1] 15% of 200 = 200 x 15 / 100\n[Step 2] 200 x 15 = 3000\n[Step 3] 3000 / 100 = 30",
            "no", "All steps are correct"
        ),
    ]
    PROMPT_TEMPLATE = (
        "Review this step-by-step solution and judge if there is an error.\n\n"
        "{proof}\n\n"
        "Answer: Is there an error? (yes/no). Then on the next line, state your confidence 0-100."
    )

    for proof, expected_verdict, explanation in PROOFS:
        response = llm.prompt(PROMPT_TEMPLATE.format(proof=proof))
        conf = extract_confidence(response)
        verdict = "yes" if "yes" in response.lower()[:50] else "no"
        kbench.assertions.assert_true(
            verdict == expected_verdict,
            expectation=f"Correct error verdict expected: '{expected_verdict}'. Reason: {explanation}. Got: '{verdict}'"
        )
        kbench.assertions.assert_true(
            conf is not None,
            expectation="Model must provide a confidence score 0-100 alongside its error judgment."
        )

In [ ]:
ALL_MODELS = {
    # Anthropic
    "claude-opus-4-6":      kbench.llms["anthropic/claude-opus-4-6@default"],
    "claude-sonnet-4-6":    kbench.llms["anthropic/claude-sonnet-4-6@default"],
    # DeepSeek
    "deepseek-v3-2":        kbench.llms["deepseek-ai/deepseek-v3.2"],
    "deepseek-r1":          kbench.llms["deepseek-ai/deepseek-r1-0528"],
    # Google Gemini
    "gemini-3-1-pro":       kbench.llms["google/gemini-3.1-pro-preview"],
    "gemini-3-flash":       kbench.llms["google/gemini-3-flash-preview"],
    # Google Gemma
    "gemma-4-31b":          kbench.llms["google/gemma-4-31b"],
    "gemma-4-26b":          kbench.llms["google/gemma-4-26b-a4b"],
    # OpenAI
    "gpt-5-4":              kbench.llms["openai/gpt-5.4-2026-03-05"],
    "gpt-5-4-mini":         kbench.llms["openai/gpt-5.4-mini-2026-03-17"],
    # Qwen
    "qwen3-235b":           kbench.llms["qwen/qwen3-235b-a22b-instruct-2507"],
    "qwen3-coder-480b":     kbench.llms["qwen/qwen3-coder-480b-a35b-instruct"],
    # ZhipuAI
    "glm-5":                kbench.llms["zai/glm-5"],
}

METACAL_TASKS = [
    t01_graded_confidence,
    t02_domain_shift_probe,
    t03_uncertainty_injection,
    t04_post_answer_error_flag,
    t05_injected_error_detection,
    t06_contradiction_detection,
    t07_accuracy_matched_discrimination,
    t08_confabulation_vs_correction,
    t09_thinking_path_quality,
    t10_hallucination_abstention,
    t11_logical_consistency,
    t12_abstention,
    t13_strategy_selection,
    t14_difficulty_prediction,
    t15_confidence_update
]

In [20]:
# Run pilot first (5 models). Once it passes, swap to ALL_MODELS.
PILOT = {k: ALL_MODELS[k] for k in [
    "claude-opus-4-6",
    "gemini-3-1-pro",
    "gpt-5-4",
    "deepseek-r1",
    "gemma-4-31b",
]}

for task in METACAL_TASKS:
    for name, model in PILOT.items():
        print(f"▶ {task.name} × {name}")
        task.run(model)

# Full sweep — uncomment when pilot passes:
# for task in METACAL_TASKS:
#     for name, model in ALL_MODELS.items():
#         task.run(model)

# Leaderboard submission — final cell only:
# %choose t01_graded_confidence

▶ T-09: Thinking Path Quality × claude-opus-4-6
▶ T-09: Thinking Path Quality × gemini-3-1-pro
▶ T-09: Thinking Path Quality × gpt-5-4
▶ T-09: Thinking Path Quality × deepseek-r1
▶ T-09: Thinking Path Quality × gemma-4-31b
